In [5]:
import numpy as np
import networkx as nx
from scipy.stats import ks_2samp

In [2]:
def compute_2d_ks(test_points, gen_points):
    ks_x = ks_2samp(test_points[:, 0], gen_points[:, 0]).statistic
    ks_y = ks_2samp(test_points[:, 1], gen_points[:, 1]).statistic
    return (ks_x + ks_y) / 2

In [ ]:
# -------- Node Degree Behavior Metric (2D KS using Q1 and Q3 of degree) -------- #
def extract_degree_q1_q3_points(graph_seq):
    q1_q3_points = []
    # print(f'graph_seq: {graph_seq}')
    for g in graph_seq:
        # print(f'g: {g}')
        # print(f'g.nodes: {g.nodes()}')
        # print(f'g.edges: {g.edges()}')
        degrees = np.array([d for _, d in g.degree()])
        if len(degrees) < 4:
            continue
        q1 = np.percentile(degrees, 25)
        q3 = np.percentile(degrees, 75)
        q1_q3_points.append([q1, q3])
    return np.array(q1_q3_points)

In [4]:
def run_node_degree_behavior_eval(test_graph_seqs, gen_graph_seqs):
    # print(test_graphs[0])
    # print(gen_graphs[0])
    n = min(len(test_graph_seqs), len(gen_graph_seqs))
    ks_scores = []
    for i in range(n):
        # print(f'test_graphs[{i}]: {test_graphs[i]}')
        test_points = extract_degree_q1_q3_points(test_graph_seqs[i])
        gen_points = extract_degree_q1_q3_points(gen_graph_seqs[i])
        if len(test_points) == 0 or len(gen_points) == 0:
            continue
        ks = compute_2d_ks(test_points, gen_points)
        ks_scores.append(ks)
    return np.mean(ks_scores) if ks_scores else None

In [10]:
# -------- Node Degree Behavior Metric (2D KS using Q1 and Q3 of degree) -------- #
def extract_centrality_q1_q3_points(graph_seq, type):
    q1_q3_points = []
    for g in graph_seq:
        if type == 'degree':
            centralities = np.array([d for _, d in nx.degree_centrality(g).items()])
        elif type == 'betweenness':
            centralities = np.array([d for _, d in nx.betweenness_centrality(g).items()])
        elif type == 'closeness':
            centralities = np.array([d for _, d in nx.closeness_centrality(g).items()])
        elif type == 'eigenvector':
            centralities = np.array([d for _, d in nx.eigenvector_centrality(g).items()])
        elif type == 'information':
            centralities = np.array([d for _, d in nx.information_centrality(g).items()])
        
        if len(centralities) < 4:
            continue
        q1 = np.percentile(centralities, 25)
        q3 = np.percentile(centralities, 75)
        q1_q3_points.append([q1, q3])
    return np.array(q1_q3_points)

In [7]:
def run_centrality_behavior_eval(test_graph_seqs, gen_graph_seqs, type):
    # print(test_graphs[0])
    # print(gen_graphs[0])
    n = min(len(test_graph_seqs), len(gen_graph_seqs))
    ks_scores = []
    for i in range(n):
        # print(f'test_graphs[{i}]: {test_graphs[i]}')
        test_points = extract_centrality_q1_q3_points(test_graph_seqs[i], type=type)
        gen_points = extract_centrality_q1_q3_points(gen_graph_seqs[i], type=type)
        if len(test_points) == 0 or len(gen_points) == 0:
            continue
        ks = compute_2d_ks(test_points, gen_points)
        ks_scores.append(ks)
    return np.mean(ks_scores) if ks_scores else None

In [8]:
def centrality_KS_evaluator(generated_graph_seqs, reference_graph_seqs, type):
    print(f'Computing {type} centrality KS...')
    centrality_bahavior_ks = run_centrality_behavior_eval(generated_graph_seqs, reference_graph_seqs, type=type)
    return centrality_bahavior_ks

In [13]:
import pickle

dataset_list = ['digg','superuser','twitter','wiki-vote']
model_list = ['TagGen','AGE','DAMNET','DYMOND'] 
centrality_types = ['degree','betweenness','closeness']
for dataset_name in dataset_list:
    print(f'dataset: {dataset_name}')
    for model_name in model_list:
        print(f'model: {model_name}')
        print('Loading testing and sampled graph data...')
        test_graph_path = f'../test_and_generated_graphs/{dataset_name}/{model_name}/test_graphs.pkl' # fill in the corresponding info
        sampled_graph_path = f'../test_and_generated_graphs/{dataset_name}/{model_name}/sampled_ts.pkl' # fill in the corresponding info
        with open(sampled_graph_path,'rb') as f_gen:
            sampled_graphs = pickle.load(f_gen)
            print(f'There are {len(sampled_graphs)} generated temporal graph sequences of length {len(sampled_graphs[0])}.')
        with open(test_graph_path,'rb') as f_test:
            test_graphs = pickle.load(f_test)
            print(f'There are {len(test_graphs)} testing temporal graph sequences of length {len(test_graphs[0])}.')
        
        num_seqs = min(len(test_graphs), len(sampled_graphs))
        for type in centrality_types:
            centrality_bahavior_ks = centrality_KS_evaluator(sampled_graphs[0:num_seqs], test_graphs[0:num_seqs], type)
            print(f'{type} centrality KS: {centrality_bahavior_ks}')

dataset: digg
model: TagGen
Loading testing and sampled graph data...
There are 5 generated temporal graph sequences of length 4.
There are 25000 testing temporal graph sequences of length 4.
Computing degree centrality KS...
degree centrality KS: 0.75
Computing betweenness centrality KS...
betweenness centrality KS: 0.725
Computing closeness centrality KS...
closeness centrality KS: 0.575
model: AGE
Loading testing and sampled graph data...
There are 25000 generated temporal graph sequences of length 4.
There are 25000 testing temporal graph sequences of length 4.
Computing degree centrality KS...
degree centrality KS: 0.792265
Computing betweenness centrality KS...
betweenness centrality KS: 0.963465
Computing closeness centrality KS...
closeness centrality KS: 0.708515
model: DAMNET
Loading testing and sampled graph data...
There are 25000 generated temporal graph sequences of length 4.
There are 25000 testing temporal graph sequences of length 4.
Computing degree centrality KS...
d

In [16]:
dataset_list = ['superuser']
model_name = 'MulDyDiff'
centrality_types = ['degree','betweenness','closeness']
for dataset_name in dataset_list:
    print(f'dataset: {dataset_name}')
    print('Loading testing and sampled graph data...')
    test_graph_path = f'../test_and_generated_graphs/{dataset_name}/{model_name}/test_graphs.pkl' # fill in the corresponding info
    sampled_graph_path = f'../test_and_generated_graphs/{dataset_name}/{model_name}/sampled_ts.pkl' # fill in the corresponding info
    with open(sampled_graph_path,'rb') as f_gen:
        sampled_graphs = pickle.load(f_gen)
        print(f'There are {len(sampled_graphs)} generated temporal graph sequences of length {len(sampled_graphs[0])}.')
    with open(test_graph_path,'rb') as f_test:
        test_graphs = pickle.load(f_test)
        print(f'There are {len(test_graphs)} testing temporal graph sequences of length {len(test_graphs[0])}.')
    num_seqs = min(len(test_graphs), len(sampled_graphs))
    for type in centrality_types:
        centrality_bahavior_ks = centrality_KS_evaluator(sampled_graphs[0:num_seqs], test_graphs[0:num_seqs], type)
        print(f'{type} centrality KS: {centrality_bahavior_ks}')

dataset: superuser
Loading testing and sampled graph data...
There are 1600 generated temporal graph sequences of length 4.
There are 16000 testing temporal graph sequences of length 4.
Computing degree centrality KS...
degree centrality KS: 0.64109375
Computing betweenness centrality KS...
betweenness centrality KS: 0.611875
Computing closeness centrality KS...
closeness centrality KS: 0.812734375
